# HLA 19-AA Anchor-Position Held-Out Amino-Acid Binding Experiment

This notebook uses `hla_only.txt`, selects one HLA allele plus one held-out standard amino acid, trains on same-allele 9-mers that do not contain that amino acid anywhere, and evaluates on same-allele 9-mers where the held-out amino acid appears at anchor positions P2/P9. The default evaluation mode is strict: the held-out amino acid may occur only at P2 and/or P9.

The compared encoders are AE centroids from `peptide_autoencoder_v3_512.pth`, 20-symbol one-hot, and BLOSUM62.


In [ ]:
from __future__ import annotations

import csv
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Callable, Iterable, Optional


PROJECT_DIR = Path(".")
DEFAULT_HLA_FILE = PROJECT_DIR / "hla_only.txt"
DEFAULT_AE_WEIGHTS = PROJECT_DIR / "peptide_autoencoder_v3_512.pth"
DEFAULT_IMAGE_DIR = PROJECT_DIR / "peptide_rotamers"
DEFAULT_OUTPUT_DIR = PROJECT_DIR / "hla_19aa_heldout_results"

STANDARD_AA = tuple("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)
AA_TO_INDEX_20 = {aa: idx for idx, aa in enumerate(STANDARD_AA)}

RANDOM_SEED = 7
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
FOLDS = 5


@dataclass(frozen=True)
class Record:
    peptide: str
    label: int
    allele: str
    fold: Optional[int] = None


def safe_token(value: str) -> str:
    return value.replace(":", "_").replace("/", "_").replace("\\", "_")


## Data and Split Helpers


In [ ]:
def read_hla_only(path: Path) -> tuple[dict[str, list[Record]], Counter]:
    grouped: dict[str, dict[str, int]] = defaultdict(dict)
    stats = Counter()

    with open(path, encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) < 3:
                stats["malformed"] += 1
                continue

            peptide, label_text, allele = parts[0].strip(), parts[1].strip(), parts[2].strip()
            try:
                label = int(label_text)
            except ValueError:
                stats["bad_label"] += 1
                continue

            if label not in {0, 1}:
                stats["bad_label"] += 1
                continue
            if len(peptide) != 9:
                stats["not_9mer"] += 1
                continue
            if any(aa not in STANDARD_AA_SET for aa in peptide):
                stats["non_standard_residue"] += 1
                continue

            previous = grouped[allele].get(peptide)
            if previous is not None and previous != label:
                stats["label_conflict_collapsed_to_positive"] += 1
            grouped[allele][peptide] = max(previous, label) if previous is not None else label
            stats["valid_rows"] += 1

    records_by_allele = {
        allele: [Record(peptide=peptide, label=label, allele=allele) for peptide, label in sorted(peptides.items())]
        for allele, peptides in sorted(grouped.items())
    }
    stats["alleles"] = len(records_by_allele)
    stats["unique_allele_peptides"] = sum(len(rows) for rows in records_by_allele.values())
    return records_by_allele, stats


def label_counts(records: Iterable[Record]) -> Counter:
    return Counter(row.label for row in records)


def heldout_positions(peptide: str, held_out_aa: str) -> list[int]:
    return [idx + 1 for idx, aa in enumerate(peptide) if aa == held_out_aa]


def is_anchor_evaluation_peptide(
    peptide: str,
    held_out_aa: str,
    anchor_positions: tuple[int, ...],
    eval_position_mode: str,
) -> bool:
    positions = heldout_positions(peptide, held_out_aa)
    if not positions:
        return False
    anchor_set = set(anchor_positions)
    has_anchor = any(position in anchor_set for position in positions)
    if eval_position_mode == "anchor_contains":
        return has_anchor
    if eval_position_mode == "anchor_only":
        return has_anchor and all(position in anchor_set for position in positions)
    raise ValueError("eval_position_mode must be 'anchor_only' or 'anchor_contains'")


def split_candidate_row(
    allele: str,
    held_out_aa: str,
    allele_records: list[Record],
    train_records: list[Record],
    eval_records: list[Record],
    anchor_positions: tuple[int, ...],
    eval_position_mode: str,
    min_train_rows: int,
    min_eval_rows: int,
    min_train_per_class: int,
    min_eval_per_class: int,
) -> dict[str, object]:
    train_counts = label_counts(train_records)
    eval_counts = label_counts(eval_records)
    all_heldout_records = [row for row in allele_records if held_out_aa in row.peptide]
    anchor_set = set(anchor_positions)
    eval_elsewhere = sum(
        any(position not in anchor_set for position in heldout_positions(row.peptide, held_out_aa))
        for row in eval_records
    )
    eval_p2 = sum(row.peptide[1] == held_out_aa for row in eval_records)
    eval_p9 = sum(row.peptide[8] == held_out_aa for row in eval_records)
    eval_both_p2_p9 = sum(row.peptide[1] == held_out_aa and row.peptide[8] == held_out_aa for row in eval_records)
    valid = (
        len(train_records) >= min_train_rows
        and len(eval_records) >= min_eval_rows
        and train_counts.get(0, 0) >= min_train_per_class
        and train_counts.get(1, 0) >= min_train_per_class
        and eval_counts.get(0, 0) >= min_eval_per_class
        and eval_counts.get(1, 0) >= min_eval_per_class
    )
    return {
        "Allele": allele,
        "Held_Out_AA": held_out_aa,
        "Anchor_Positions": ",".join(str(position) for position in anchor_positions),
        "Eval_Position_Mode": eval_position_mode,
        "Allele_Total": len(allele_records),
        "All_Heldout_Rows": len(all_heldout_records),
        "Train_Rows": len(train_records),
        "Train_Positive": train_counts.get(1, 0),
        "Train_Negative": train_counts.get(0, 0),
        "Eval_Rows": len(eval_records),
        "Eval_Positive": eval_counts.get(1, 0),
        "Eval_Negative": eval_counts.get(0, 0),
        "Eval_With_Heldout_At_P2": eval_p2,
        "Eval_With_Heldout_At_P9": eval_p9,
        "Eval_With_Heldout_At_Both_P2_P9": eval_both_p2_p9,
        "Eval_With_Heldout_Outside_Anchors": eval_elsewhere,
        "Anchor_Eval_Fraction_Of_All_Heldout": len(eval_records) / len(all_heldout_records) if all_heldout_records else 0.0,
        "Min_Positive_Support": min(train_counts.get(1, 0), eval_counts.get(1, 0)),
        "Min_Class_Support": min(train_counts.get(0, 0), train_counts.get(1, 0), eval_counts.get(0, 0), eval_counts.get(1, 0)),
        "Train_Eval_Total": len(train_records) + len(eval_records),
        "Valid": valid,
    }


def compute_split_candidates(
    records_by_allele: dict[str, list[Record]],
    anchor_positions: tuple[int, ...],
    eval_position_mode: str,
    min_train_rows: int,
    min_eval_rows: int,
    min_train_per_class: int,
    min_eval_per_class: int,
) -> list[dict[str, object]]:
    candidates = []
    for allele, records in records_by_allele.items():
        for held_out_aa in STANDARD_AA:
            train_records = [row for row in records if held_out_aa not in row.peptide]
            eval_records = [
                row for row in records
                if is_anchor_evaluation_peptide(row.peptide, held_out_aa, anchor_positions, eval_position_mode)
            ]
            candidates.append(
                split_candidate_row(
                    allele,
                    held_out_aa,
                    records,
                    train_records,
                    eval_records,
                    anchor_positions,
                    eval_position_mode,
                    min_train_rows,
                    min_eval_rows,
                    min_train_per_class,
                    min_eval_per_class,
                )
            )
    return candidates


def candidate_sort_key(row: dict[str, object], ranking: str = "train_eval_total") -> tuple[object, ...]:
    common_tail = (
        int(row["Min_Class_Support"]),
        int(row["Train_Positive"]),
        int(row["Eval_Positive"]),
        str(row["Allele"]),
        str(row["Held_Out_AA"]),
    )
    if ranking == "eval_rows":
        primary = (int(row["Eval_Rows"]), int(row["Train_Rows"]), int(row["Train_Eval_Total"]))
    elif ranking == "balanced_positive_support":
        primary = (int(row["Min_Positive_Support"]), int(row["Eval_Rows"]), int(row["Train_Rows"]))
    elif ranking == "train_eval_total":
        primary = (int(row["Train_Eval_Total"]), int(row["Train_Rows"]), int(row["Eval_Rows"]))
    else:
        raise ValueError("ranking must be 'train_eval_total', 'eval_rows', or 'balanced_positive_support'")
    return (int(row["Valid"]),) + primary + common_tail


def choose_candidate(
    candidates: list[dict[str, object]],
    allele: Optional[str],
    held_out_aa: Optional[str],
    ranking: str,
) -> dict[str, object]:
    filtered = candidates
    if allele:
        filtered = [row for row in filtered if row["Allele"] == allele]
    if held_out_aa:
        filtered = [row for row in filtered if row["Held_Out_AA"] == held_out_aa]

    valid = [row for row in filtered if row["Valid"]]
    if not valid:
        filters = []
        if allele:
            filters.append(f"allele={allele}")
        if held_out_aa:
            filters.append(f"held_out_aa={held_out_aa}")
        suffix = f" for {' and '.join(filters)}" if filters else ""
        raise ValueError(f"No valid anchor-position held-out split found{suffix}. Lower thresholds or inspect the candidate CSV.")

    return dict(max(valid, key=lambda row: candidate_sort_key(row, ranking=ranking)))


def build_selected_split(
    records_by_allele: dict[str, list[Record]],
    selected: dict[str, object],
    anchor_positions: tuple[int, ...],
    eval_position_mode: str,
    folds: int,
    seed: int,
) -> tuple[list[Record], list[Record]]:
    allele = str(selected["Allele"])
    held_out_aa = str(selected["Held_Out_AA"])
    allele_records = records_by_allele[allele]
    train_records = [row for row in allele_records if held_out_aa not in row.peptide]
    eval_records = [
        row for row in allele_records
        if is_anchor_evaluation_peptide(row.peptide, held_out_aa, anchor_positions, eval_position_mode)
    ]
    cv_records = assign_stratified_folds(train_records, folds=folds, seed=seed)

    assert all(row.allele == allele for row in cv_records)
    assert all(row.allele == allele for row in eval_records)
    assert all(held_out_aa not in row.peptide for row in cv_records)
    assert all(is_anchor_evaluation_peptide(row.peptide, held_out_aa, anchor_positions, eval_position_mode) for row in eval_records)
    if eval_position_mode == "anchor_only":
        anchor_set = set(anchor_positions)
        assert all(
            all(position in anchor_set for position in heldout_positions(row.peptide, held_out_aa))
            for row in eval_records
        )
    assert not ({row.peptide for row in cv_records} & {row.peptide for row in eval_records})
    return cv_records, eval_records


def assign_stratified_folds(records: list[Record], folds: int, seed: int) -> list[Record]:
    by_label: dict[int, list[Record]] = defaultdict(list)
    for row in sorted(records, key=lambda item: item.peptide):
        by_label[row.label].append(row)

    for label in (0, 1):
        if len(by_label[label]) < folds:
            raise ValueError(f"Need at least {folds} rows for class {label} in the train/test split.")

    rng = random.Random(seed)
    folded = []
    for label, label_records in sorted(by_label.items()):
        shuffled = list(label_records)
        rng.shuffle(shuffled)
        for idx, row in enumerate(shuffled):
            folded.append(replace(row, fold=idx % folds))

    return sorted(folded, key=lambda item: (item.fold if item.fold is not None else -1, item.peptide))


def write_split_candidates(rows: list[dict[str, object]], path: Path, ranking: str) -> None:
    fieldnames = [
        "Allele",
        "Held_Out_AA",
        "Anchor_Positions",
        "Eval_Position_Mode",
        "Allele_Total",
        "All_Heldout_Rows",
        "Train_Rows",
        "Train_Positive",
        "Train_Negative",
        "Eval_Rows",
        "Eval_Positive",
        "Eval_Negative",
        "Eval_With_Heldout_At_P2",
        "Eval_With_Heldout_At_P9",
        "Eval_With_Heldout_At_Both_P2_P9",
        "Eval_With_Heldout_Outside_Anchors",
        "Anchor_Eval_Fraction_Of_All_Heldout",
        "Min_Positive_Support",
        "Min_Class_Support",
        "Train_Eval_Total",
        "Valid",
    ]
    sorted_rows = sorted(rows, key=lambda row: candidate_sort_key(row, ranking=ranking), reverse=True)
    save_rows_csv(sorted_rows, path, fieldnames)


def write_records_csv(records: list[Record], path: Path, include_fold: bool) -> None:
    fieldnames = ["Peptide", "Label", "Allele"]
    if include_fold:
        fieldnames.append("Fold")
    rows = []
    for row in records:
        out = {"Peptide": row.peptide, "Label": row.label, "Allele": row.allele}
        if include_fold:
            out["Fold"] = row.fold
        rows.append(out)
    save_rows_csv(rows, path, fieldnames)


def write_labeled_txt(records: list[Record], path: Path, include_fold: bool) -> None:
    with open(path, "w", encoding="utf-8", newline="") as handle:
        for row in records:
            if include_fold:
                handle.write(f"{row.peptide} {row.label} {row.allele} {row.fold}\n")
            else:
                handle.write(f"{row.peptide} {row.label} {row.allele}\n")


def save_rows_csv(rows: list[dict[str, object]], path: Path, fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


## Encoders


In [ ]:
def one_hot_20_featurizer(sequence: str):
    import numpy as np

    matrix = np.zeros((len(sequence), len(STANDARD_AA)), dtype=np.float32)
    for position, aa in enumerate(sequence):
        matrix[position, AA_TO_INDEX_20[aa]] = 1.0
    return matrix.reshape(-1)


BLOSUM62_ORDER = tuple("ARNDCQEGHILKMFPSTWYV")
BLOSUM62_VALUES = [
    [4, -1, -2, -2, 0, -1, -1, 0, -2, -1, -1, -1, -1, -2, -1, 1, 0, -3, -2, 0],
    [-1, 5, 0, -2, -3, 1, 0, -2, 0, -3, -2, 2, -1, -3, -2, -1, -1, -3, -2, -3],
    [-2, 0, 6, 1, -3, 0, 0, 0, 1, -3, -3, 0, -2, -3, -2, 1, 0, -4, -2, -3],
    [-2, -2, 1, 6, -3, 0, 2, -1, -1, -3, -4, -1, -3, -3, -1, 0, -1, -4, -3, -3],
    [0, -3, -3, -3, 9, -3, -4, -3, -3, -1, -1, -3, -1, -2, -3, -1, -1, -2, -2, -1],
    [-1, 1, 0, 0, -3, 5, 2, -2, 0, -3, -2, 1, 0, -3, -1, 0, -1, -2, -1, -2],
    [-1, 0, 0, 2, -4, 2, 5, -2, 0, -3, -3, 1, -2, -3, -1, 0, -1, -3, -2, -2],
    [0, -2, 0, -1, -3, -2, -2, 6, -2, -4, -4, -2, -3, -3, -2, 0, -2, -2, -3, -3],
    [-2, 0, 1, -1, -3, 0, 0, -2, 8, -3, -3, -1, -2, -1, -2, -1, -2, -2, 2, -3],
    [-1, -3, -3, -3, -1, -3, -3, -4, -3, 4, 2, -3, 1, 0, -3, -2, -1, -3, -1, 3],
    [-1, -2, -3, -4, -1, -2, -3, -4, -3, 2, 4, -2, 2, 0, -3, -2, -1, -2, -1, 1],
    [-1, 2, 0, -1, -3, 1, 1, -2, -1, -3, -2, 5, -1, -3, -1, 0, -1, -3, -2, -2],
    [-1, -1, -2, -3, -1, 0, -2, -3, -2, 1, 2, -1, 5, 0, -2, -1, -1, -1, -1, 1],
    [-2, -3, -3, -3, -2, -3, -3, -3, -1, 0, 0, -3, 0, 6, -4, -2, -2, 1, 3, -1],
    [-1, -2, -2, -1, -3, -1, -1, -2, -2, -3, -3, -1, -2, -4, 7, -1, -1, -4, -3, -2],
    [1, -1, 1, 0, -1, 0, 0, 0, -1, -2, -2, 0, -1, -2, -1, 4, 1, -3, -2, -2],
    [0, -1, 0, -1, -1, -1, -1, -2, -2, -1, -1, -1, -1, -2, -1, 1, 5, -2, -2, 0],
    [-3, -3, -4, -4, -2, -2, -3, -2, -2, -3, -2, -3, -1, 1, -4, -3, -2, 11, 2, -3],
    [-2, -2, -2, -3, -2, -1, -2, -3, 2, -1, -1, -2, -1, 3, -3, -2, -2, 2, 7, -1],
    [0, -3, -3, -3, -1, -2, -2, -3, -3, 3, 1, -2, 1, -1, -2, -2, 0, -3, -1, 4],
]
BLOSUM62_INDEX = {aa: idx for idx, aa in enumerate(BLOSUM62_ORDER)}


def blosum62_20_featurizer(sequence: str):
    import numpy as np

    rows = [BLOSUM62_VALUES[BLOSUM62_INDEX[aa]] for aa in sequence]
    return np.array(rows, dtype=np.float32).reshape(-1)


AA_TO_IMAGE_CLASS = {
    "alanine": "A",
    "arginine": "R",
    "asparagine": "N",
    "aspartic_acid": "D",
    "cysteine": "C",
    "glutamic_acid": "E",
    "glutamine": "Q",
    "glycine": "G",
    "histidine": "H",
    "isoleucine": "I",
    "leucine": "L",
    "lysine": "K",
    "methionine": "M",
    "phenylalanine": "F",
    "proline": "P",
    "serine": "S",
    "threonine": "T",
    "tryptophan": "W",
    "tyrosine": "Y",
    "valine": "V",
}


def load_autoencoder_centroids(weights_path: Path, image_dir: Path, device):
    if not weights_path.exists():
        raise FileNotFoundError(f"Missing AE weights: {weights_path}")
    if not image_dir.exists():
        raise FileNotFoundError(
            f"Missing AE image directory: {image_dir}. "
            "The AE encoder needs the same residue rotamer image folders used by SIHA_ex.ipynb."
        )

    import numpy as np
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms

    class SEBlock(nn.Module):
        def __init__(self, channels: int, reduction: int = 16):
            super().__init__()
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.fc = nn.Sequential(
                nn.Linear(channels, channels // reduction),
                nn.ReLU(inplace=True),
                nn.Linear(channels // reduction, channels),
                nn.Sigmoid(),
            )

        def forward(self, x):
            batch, channels, _, _ = x.size()
            y = self.pool(x).view(batch, channels)
            y = self.fc(y).view(batch, channels, 1, 1)
            return x * y

    class AutoEncoderV3(nn.Module):
        def __init__(self, bottleneck_size: int = 512):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(3, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2),
                SEBlock(32),
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2),
                SEBlock(64),
                nn.Conv2d(64, 128, kernel_size=3, padding=1),
                nn.BatchNorm2d(128),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2),
                SEBlock(128),
                nn.Conv2d(128, 256, kernel_size=3, padding=1),
                nn.BatchNorm2d(256),
                nn.LeakyReLU(0.1),
                nn.MaxPool2d(2),
                SEBlock(256),
                nn.Flatten(),
                nn.Linear(256 * 15 * 15, bottleneck_size),
            )

        def forward(self, x):
            return self.encoder(x)

    print(f"Loading AE weights: {weights_path}")
    model = AutoEncoderV3(bottleneck_size=512).to(device)
    try:
        state_dict = torch.load(weights_path, map_location=device, weights_only=True)
    except TypeError:
        state_dict = torch.load(weights_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    dataset = datasets.ImageFolder(str(image_dir), transform=transforms.ToTensor())
    loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0)
    latent_vectors = []
    labels = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            if device.type == "cuda":
                with torch.autocast(device_type="cuda"):
                    vectors = model.encoder(inputs)
            else:
                vectors = model.encoder(inputs)
            latent_vectors.append(vectors.cpu().numpy())
            labels.extend(targets.numpy())

    latent_matrix = np.concatenate(latent_vectors, axis=0)
    labels = np.array(labels)
    centroids = {}
    for idx, class_name in enumerate(dataset.classes):
        letter = AA_TO_IMAGE_CLASS.get(class_name)
        if letter is None:
            continue
        centroids[letter] = latent_matrix[labels == idx].mean(axis=0)

    missing = sorted(STANDARD_AA_SET - set(centroids))
    if missing:
        raise ValueError(f"Missing AE centroid(s) for standard residues: {missing}")
    print(f"Loaded AE centroids for {len(centroids)} standard residues.")
    return centroids


def make_autoencoder_featurizer(centroid_dict: dict[str, object]) -> Callable[[str], object]:
    def featurize(sequence: str):
        import numpy as np

        return np.concatenate([centroid_dict[aa] for aa in sequence]).astype(np.float32)

    return featurize



## Training and Metrics Helpers


In [ ]:
def records_to_tensors(records: list[Record], featurize: Callable[[str], object]):
    import numpy as np
    import torch

    features = [featurize(row.peptide) for row in records]
    labels = [row.label for row in records]
    x = torch.tensor(np.array(features), dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
    return x, y


def run_fivefold_cv(
    records: list[Record],
    featurize: Callable[[str], object],
    experiment: dict[str, object],
    device,
    epochs: int,
    batch_size: int,
    lr: float,
    seed: int,
) -> list[object]:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset

    class TorchLinearNN(nn.Module):
        def __init__(self, input_size: int):
            super().__init__()
            self.fc = nn.Linear(input_size, 1)
            # Important for one-hot held-out residues: columns absent during
            # training remain neutral instead of retaining random weights.
            nn.init.zeros_(self.fc.weight)
            nn.init.zeros_(self.fc.bias)

        def forward(self, x):
            return self.fc(x)

    all_results = []
    trained_models = []
    for fold_idx in range(int(experiment["folds"])):
        train_records = [row for row in records if row.fold != fold_idx]
        test_records = [row for row in records if row.fold == fold_idx]
        print(
            f"--- {experiment['id']} fold {fold_idx}: "
            f"train={len(train_records)} test={len(test_records)} ---"
        )

        x_train, y_train = records_to_tensors(train_records, featurize)
        x_test, y_test = records_to_tensors(test_records, featurize)

        generator = torch.Generator()
        generator.manual_seed(seed + fold_idx)
        train_loader = DataLoader(
            TensorDataset(x_train, y_train),
            batch_size=batch_size,
            shuffle=True,
            generator=generator,
        )

        model = TorchLinearNN(input_size=x_train.shape[1]).to(device)
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        model.train()
        for _ in range(epochs):
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(batch_x), batch_y)
                loss.backward()
                optimizer.step()

        model_path = Path(experiment["output_dir"]) / f"{experiment['model_prefix']}_fold_{fold_idx}.pth"
        torch.save(model.state_dict(), model_path)
        trained_models.append(model)

        model.eval()
        probabilities = []
        with torch.no_grad():
            for start in range(0, len(x_test), batch_size):
                batch_x = x_test[start : start + batch_size].to(device)
                probabilities.extend(torch.sigmoid(model(batch_x)).cpu().numpy().flatten().tolist())

        for row, true_value, probability in zip(test_records, y_test.numpy().flatten(), probabilities):
            all_results.append(
                {
                    "Experiment": experiment["id"],
                    "Encoder": experiment["encoder_id"],
                    "Allele": experiment["allele"],
                    "Held_Out_AA": experiment["held_out_aa"],
                    "Fold": fold_idx,
                    "Peptide": row.peptide,
                    "True_Class": int(true_value),
                    "Probability": float(probability),
                }
            )

    fieldnames = [
        "Experiment",
        "Encoder",
        "Allele",
        "Held_Out_AA",
        "Fold",
        "Peptide",
        "True_Class",
        "Probability",
    ]
    save_rows_csv(all_results, Path(experiment["cv_csv"]), fieldnames)
    print(f"Saved CV predictions: {experiment['cv_csv']} | rows={len(all_results)}")
    return trained_models


def predict_heldout_evaluation(
    records: list[Record],
    featurize: Callable[[str], object],
    models: list[object],
    experiment: dict[str, object],
    device,
    batch_size: int,
) -> list[dict[str, object]]:
    import numpy as np
    import torch

    x, y = records_to_tensors(records, featurize)
    predictions_matrix = np.zeros((len(x), len(models)), dtype=np.float32)
    x = x.to(device)

    for model_idx, model in enumerate(models):
        model.eval()
        fold_probabilities = []
        with torch.no_grad():
            for start in range(0, len(x), batch_size):
                batch_x = x[start : start + batch_size]
                fold_probabilities.extend(torch.sigmoid(model(batch_x)).cpu().numpy().flatten().tolist())
        predictions_matrix[:, model_idx] = fold_probabilities

    ensemble = predictions_matrix.mean(axis=1)
    rows = []
    for idx, (row, true_value, mean_probability) in enumerate(zip(records, y.numpy().flatten(), ensemble)):
        out = {
            "Experiment": experiment["id"],
            "Encoder": experiment["encoder_id"],
            "Allele": experiment["allele"],
            "Held_Out_AA": experiment["held_out_aa"],
            "Index": idx,
            "Peptide": row.peptide,
            "True_Class": int(true_value),
            "Ensemble_Probability": float(mean_probability),
        }
        for model_idx in range(len(models)):
            out[f"Fold_{model_idx}_Probability"] = float(predictions_matrix[idx, model_idx])
        rows.append(out)

    fieldnames = [
        "Experiment",
        "Encoder",
        "Allele",
        "Held_Out_AA",
        "Index",
        "Peptide",
        "True_Class",
        "Ensemble_Probability",
    ] + [f"Fold_{idx}_Probability" for idx in range(len(models))]
    save_rows_csv(rows, Path(experiment["eval_csv"]), fieldnames)
    print(f"Saved held-out evaluation predictions: {experiment['eval_csv']} | rows={len(rows)}")
    return rows


def rank_auc(y_true: list[int], y_score: list[float]) -> float:
    positives = sum(1 for value in y_true if value == 1)
    negatives = sum(1 for value in y_true if value == 0)
    if positives == 0 or negatives == 0:
        return math.nan

    pairs = sorted(zip(y_score, y_true), key=lambda item: item[0])
    rank_sum_positive = 0.0
    idx = 0
    while idx < len(pairs):
        end = idx + 1
        while end < len(pairs) and pairs[end][0] == pairs[idx][0]:
            end += 1
        average_rank = (idx + 1 + end) / 2.0
        rank_sum_positive += average_rank * sum(1 for _, label in pairs[idx:end] if label == 1)
        idx = end

    return (rank_sum_positive - positives * (positives + 1) / 2.0) / (positives * negatives)


def threshold_metrics(y_true: list[int], y_score: list[float], threshold: float = 0.5) -> dict[str, float]:
    predicted = [1 if value >= threshold else 0 for value in y_score]
    tp = sum(1 for truth, pred in zip(y_true, predicted) if truth == 1 and pred == 1)
    tn = sum(1 for truth, pred in zip(y_true, predicted) if truth == 0 and pred == 0)
    fp = sum(1 for truth, pred in zip(y_true, predicted) if truth == 0 and pred == 1)
    fn = sum(1 for truth, pred in zip(y_true, predicted) if truth == 1 and pred == 0)
    total = len(y_true)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "Accuracy": (tp + tn) / total if total else math.nan,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Predicted_Positive": tp + fp,
    }


def summarize_cv_rows(rows: list[dict[str, object]], experiment: dict[str, object]) -> dict[str, object]:
    y_true = [int(row["True_Class"]) for row in rows]
    y_score = [float(row["Probability"]) for row in rows]
    fold_aucs = []
    for fold_idx in sorted({int(row["Fold"]) for row in rows}):
        fold_rows = [row for row in rows if int(row["Fold"]) == fold_idx]
        fold_aucs.append(
            rank_auc(
                [int(row["True_Class"]) for row in fold_rows],
                [float(row["Probability"]) for row in fold_rows],
            )
        )
    valid_aucs = [value for value in fold_aucs if not math.isnan(value)]
    metrics = threshold_metrics(y_true, y_score)
    return {
        "Experiment": experiment["id"],
        "Encoder": experiment["encoder_id"],
        "Allele": experiment["allele"],
        "Held_Out_AA": experiment["held_out_aa"],
        "Rows": len(rows),
        "Positives": sum(y_true),
        "Negatives": len(y_true) - sum(y_true),
        "Mean_AUC": statistics.mean(valid_aucs) if valid_aucs else math.nan,
        "Std_AUC": statistics.pstdev(valid_aucs) if len(valid_aucs) > 1 else 0.0,
        **metrics,
    }


def summarize_eval_rows(rows: list[dict[str, object]], experiment: dict[str, object]) -> dict[str, object]:
    y_true = [int(row["True_Class"]) for row in rows]
    y_score = [float(row["Ensemble_Probability"]) for row in rows]
    metrics = threshold_metrics(y_true, y_score)
    return {
        "Experiment": experiment["id"],
        "Encoder": experiment["encoder_id"],
        "Allele": experiment["allele"],
        "Held_Out_AA": experiment["held_out_aa"],
        "Rows": len(rows),
        "Positives": sum(y_true),
        "Negatives": len(y_true) - sum(y_true),
        "AUC": rank_auc(y_true, y_score),
        "Mean_Probability": statistics.mean(y_score) if y_score else math.nan,
        "Median_Probability": statistics.median(y_score) if y_score else math.nan,
        **metrics,
    }


def build_encoder_specs(args, device) -> list[dict[str, object]]:
    specs = []
    requested = set(args.encoders)

    if "ae" in requested:
        centroids = load_autoencoder_centroids(args.ae_weights, args.image_dir, device)
        specs.append(
            {
                "id": "ae",
                "label": "AE",
                "expected_dim": 9 * 512,
                "featurizer": make_autoencoder_featurizer(centroids),
            }
        )

    if "onehot20" in requested:
        specs.append(
            {
                "id": "onehot20",
                "label": "One-hot 20",
                "expected_dim": 9 * len(STANDARD_AA),
                "featurizer": one_hot_20_featurizer,
            }
        )

    if "blosum62_20" in requested:
        specs.append(
            {
                "id": "blosum62_20",
                "label": "BLOSUM62 20",
                "expected_dim": 9 * len(BLOSUM62_ORDER),
                "featurizer": blosum62_20_featurizer,
            }
        )

    return specs


def make_experiment(
    encoder_spec: dict[str, object],
    selected: dict[str, object],
    output_dir: Path,
    folds: int,
) -> dict[str, object]:
    allele = str(selected["Allele"])
    held_out_aa = str(selected["Held_Out_AA"])
    encoder_id = str(encoder_spec["id"])
    split_token = str(selected.get("Split_Token", f"{safe_token(allele)}_holdout_{held_out_aa}"))
    experiment_id = f"{split_token}__{encoder_id}"
    return {
        "id": experiment_id,
        "encoder_id": encoder_id,
        "encoder_label": encoder_spec["label"],
        "allele": allele,
        "held_out_aa": held_out_aa,
        "folds": folds,
        "output_dir": output_dir,
        "model_prefix": f"hla_19aa_{split_token}_{encoder_id}",
        "cv_csv": output_dir / f"HLA_19AA_CV_Binding_Predictions_{experiment_id}.csv",
        "eval_csv": output_dir / f"HLA_19AA_Heldout_Evaluation_{experiment_id}.csv",
        "featurizer": encoder_spec["featurizer"],
        "expected_dim": encoder_spec["expected_dim"],
    }


def assert_feature_dimensions(experiments: list[dict[str, object]], sample_record: Record) -> None:
    for experiment in experiments:
        observed = len(experiment["featurizer"](sample_record.peptide))
        expected = int(experiment["expected_dim"])
        if observed != expected:
            raise AssertionError(f"{experiment['id']} dim mismatch: expected {expected}, observed {observed}")
    print(f"Feature dimension checks passed for {len(experiments)} encoders.")


def load_prediction_rows(path: Path) -> list[dict[str, object]]:
    with open(path, encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def summarize_fold_distribution(records: list[Record]) -> list[dict[str, object]]:
    rows = []
    for fold_idx in sorted({row.fold for row in records}):
        fold_records = [row for row in records if row.fold == fold_idx]
        counts = label_counts(fold_records)
        rows.append(
            {
                "Fold": fold_idx,
                "Rows": len(fold_records),
                "Positive": counts.get(1, 0),
                "Negative": counts.get(0, 0),
            }
        )
    return rows


## Configuration


In [ ]:
# Notebook configuration
# Set ALLELE or HELD_OUT_AA to None to automatically choose a valid split.
HLA_FILE = DEFAULT_HLA_FILE
OUTPUT_DIR = PROJECT_DIR / "hla_19aa_anchor_heldout_results"
AE_WEIGHTS = DEFAULT_AE_WEIGHTS
IMAGE_DIR = DEFAULT_IMAGE_DIR

ALLELE = None          # Example: "HLA-A29:02"
HELD_OUT_AA = None     # Example: "W"

# P2 and P9 are 1-based peptide positions.
ANCHOR_POSITIONS = (2, 9)

# "anchor_only": held-out AA may occur only at P2/P9 in evaluation peptides.
# "anchor_contains": held-out AA must occur at P2/P9, but may also occur elsewhere.
EVAL_POSITION_MODE = "anchor_only"

# "train_eval_total": largest train/test + anchor-evaluation split.
# "eval_rows": largest P2/P9 held-out evaluation set.
# "balanced_positive_support": largest minimum of train-positive and eval-positive support.
CANDIDATE_RANKING = "train_eval_total"

MIN_TRAIN_ROWS = 1000
MIN_EVAL_ROWS = 100
MIN_TRAIN_PER_CLASS = 25
MIN_EVAL_PER_CLASS = 10

FOLDS = 5
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 7
ENCODERS = ["ae", "onehot20", "blosum62_20"]
PREPARE_ONLY = False

if HELD_OUT_AA is not None:
    HELD_OUT_AA = HELD_OUT_AA.upper()
    if HELD_OUT_AA not in STANDARD_AA_SET:
        raise ValueError(f"HELD_OUT_AA must be one of {''.join(STANDARD_AA)}")
if EVAL_POSITION_MODE not in {"anchor_only", "anchor_contains"}:
    raise ValueError("EVAL_POSITION_MODE must be 'anchor_only' or 'anchor_contains'")
if CANDIDATE_RANKING not in {"train_eval_total", "eval_rows", "balanced_positive_support"}:
    raise ValueError("CANDIDATE_RANKING must be 'train_eval_total', 'eval_rows', or 'balanced_positive_support'")
if any(position < 1 or position > 9 for position in ANCHOR_POSITIONS):
    raise ValueError("ANCHOR_POSITIONS must be 1-based positions from 1 to 9")

args = SimpleNamespace(
    hla_file=HLA_FILE,
    output_dir=OUTPUT_DIR,
    ae_weights=AE_WEIGHTS,
    image_dir=IMAGE_DIR,
    allele=ALLELE,
    held_out_aa=HELD_OUT_AA,
    anchor_positions=tuple(ANCHOR_POSITIONS),
    eval_position_mode=EVAL_POSITION_MODE,
    candidate_ranking=CANDIDATE_RANKING,
    min_train_rows=MIN_TRAIN_ROWS,
    min_eval_rows=MIN_EVAL_ROWS,
    min_train_per_class=MIN_TRAIN_PER_CLASS,
    min_eval_per_class=MIN_EVAL_PER_CLASS,
    folds=FOLDS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    encoders=ENCODERS,
    prepare_only=PREPARE_ONLY,
)
args.output_dir.mkdir(parents=True, exist_ok=True)


## Prepare Split


In [ ]:
# Prepare HLA-only anchor-position held-out amino-acid split
records_by_allele, load_stats = read_hla_only(args.hla_file)
print(f"Loaded {load_stats['valid_rows']} valid rows from {args.hla_file}")
print(f"Collapsed to {load_stats['unique_allele_peptides']} unique allele-peptide records across {load_stats['alleles']} alleles")
if load_stats["label_conflict_collapsed_to_positive"]:
    print(f"Collapsed label conflicts to positive: {load_stats['label_conflict_collapsed_to_positive']}")

candidates = compute_split_candidates(
    records_by_allele,
    anchor_positions=args.anchor_positions,
    eval_position_mode=args.eval_position_mode,
    min_train_rows=args.min_train_rows,
    min_eval_rows=args.min_eval_rows,
    min_train_per_class=args.min_train_per_class,
    min_eval_per_class=args.min_eval_per_class,
)
candidate_csv = args.output_dir / "HLA_19AA_Anchor_Position_Split_Candidates.csv"
write_split_candidates(candidates, candidate_csv, ranking=args.candidate_ranking)
print(f"Saved split candidate report: {candidate_csv}")

selected = choose_candidate(
    candidates,
    allele=args.allele,
    held_out_aa=args.held_out_aa,
    ranking=args.candidate_ranking,
)
cv_records, eval_records = build_selected_split(
    records_by_allele,
    selected,
    anchor_positions=args.anchor_positions,
    eval_position_mode=args.eval_position_mode,
    folds=args.folds,
    seed=args.seed,
)

anchor_token = "anchor" + "_".join(str(position) for position in args.anchor_positions)
mode_token = "only" if args.eval_position_mode == "anchor_only" else "contains"
split_token = f"{safe_token(str(selected['Allele']))}_holdout_{selected['Held_Out_AA']}_{anchor_token}_{mode_token}"
selected["Split_Token"] = split_token
train_csv = args.output_dir / f"HLA_19AA_{split_token}_TrainTest_9mers_Labeled.csv"
train_txt = args.output_dir / f"HLA_19AA_{split_token}_TrainTest_9mers_Labeled.txt"
eval_csv = args.output_dir / f"HLA_19AA_{split_token}_Heldout_Evaluation_9mers_Labeled.csv"
eval_txt = args.output_dir / f"HLA_19AA_{split_token}_Heldout_Evaluation_9mers_Labeled.txt"
fold_csv = args.output_dir / f"HLA_19AA_{split_token}_Fold_Distribution.csv"

write_records_csv(cv_records, train_csv, include_fold=True)
write_labeled_txt(cv_records, train_txt, include_fold=True)
write_records_csv(eval_records, eval_csv, include_fold=False)
write_labeled_txt(eval_records, eval_txt, include_fold=False)
save_rows_csv(summarize_fold_distribution(cv_records), fold_csv, ["Fold", "Rows", "Positive", "Negative"])

print("Selected anchor-position split:")
print(
    f"  allele={selected['Allele']} held_out_aa={selected['Held_Out_AA']} "
    f"anchors={selected['Anchor_Positions']} mode={selected['Eval_Position_Mode']} ranking={args.candidate_ranking}"
)
print(
    f"  train/test={len(cv_records)} positives={selected['Train_Positive']} negatives={selected['Train_Negative']} | "
    f"anchor eval={len(eval_records)} positives={selected['Eval_Positive']} negatives={selected['Eval_Negative']}"
)
print(
    f"  eval P2={selected['Eval_With_Heldout_At_P2']} P9={selected['Eval_With_Heldout_At_P9']} "
    f"both P2/P9={selected['Eval_With_Heldout_At_Both_P2_P9']} outside anchors={selected['Eval_With_Heldout_Outside_Anchors']}"
)
print(f"  anchor eval fraction of all held-out-AA peptides: {selected['Anchor_Eval_Fraction_Of_All_Heldout']:.3f}")
print(f"Saved selected train/test split: {train_csv}")
print(f"Saved selected held-out evaluation split: {eval_csv}")


## Train and Evaluate


In [ ]:
# Run the encoder comparison
if args.prepare_only:
    print("Preparation complete. Set PREPARE_ONLY = False and re-run from the config cell to train the models.")
else:
    import numpy as np
    import torch

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"System hardware: {device}")

    encoder_specs = build_encoder_specs(args, device)
    experiments = [make_experiment(spec, selected, args.output_dir, args.folds) for spec in encoder_specs]
    assert_feature_dimensions(experiments, cv_records[0])

    cv_summary_rows = []
    eval_summary_rows = []
    for experiment in experiments:
        print("\n" + "=" * 90)
        print(f"Running {experiment['id']} | encoder={experiment['encoder_label']}")
        print("=" * 90)
        models = run_fivefold_cv(
            cv_records,
            experiment["featurizer"],
            experiment,
            device,
            epochs=args.epochs,
            batch_size=args.batch_size,
            lr=args.learning_rate,
            seed=args.seed,
        )
        eval_rows = predict_heldout_evaluation(
            eval_records,
            experiment["featurizer"],
            models,
            experiment,
            device,
            batch_size=max(args.batch_size, 128),
        )
        cv_rows = load_prediction_rows(Path(experiment["cv_csv"]))
        cv_summary_rows.append(summarize_cv_rows(cv_rows, experiment))
        eval_summary_rows.append(summarize_eval_rows(eval_rows, experiment))

    cv_summary_path = args.output_dir / f"HLA_19AA_{split_token}_CV_Summary.csv"
    eval_summary_path = args.output_dir / f"HLA_19AA_{split_token}_Heldout_Evaluation_Summary.csv"
    summary_fields = [
        "Experiment",
        "Encoder",
        "Allele",
        "Held_Out_AA",
        "Rows",
        "Positives",
        "Negatives",
        "Mean_AUC",
        "Std_AUC",
        "AUC",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Predicted_Positive",
        "Mean_Probability",
        "Median_Probability",
    ]
    cv_fields = [field for field in summary_fields if any(field in row for row in cv_summary_rows)]
    eval_fields = [field for field in summary_fields if any(field in row for row in eval_summary_rows)]
    save_rows_csv(cv_summary_rows, cv_summary_path, cv_fields)
    save_rows_csv(eval_summary_rows, eval_summary_path, eval_fields)
    print(f"Saved CV summary: {cv_summary_path}")
    print(f"Saved held-out evaluation summary: {eval_summary_path}")


## Summary Tables


In [ ]:
# Inspect summaries after training
try:
    import pandas as pd
    from IPython.display import display

    if "cv_summary_path" in globals() and Path(cv_summary_path).exists():
        print("CV summary")
        display(pd.read_csv(cv_summary_path).sort_values("Mean_AUC", ascending=False))
    if "eval_summary_path" in globals() and Path(eval_summary_path).exists():
        print("Held-out amino-acid evaluation summary")
        display(pd.read_csv(eval_summary_path).sort_values("AUC", ascending=False))
except ImportError:
    print("pandas/IPython display is not available; summary CSVs were still written to disk.")
